In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# 1. Point directly to your installed JDK 17 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Ensure PySpark uses the active Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Now import and start Spark


spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[1]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .getOrCreate()

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [4]:
df_categories =df_read_business_data.select(F.col("categories"))

In [6]:
@F.udf(returnType=StringType())
def get_category_name(text):

    if text is None:
       return "Not Specified", ["Not Specified"]
    
    list_of_element = text.split(",")
    category_name=list_of_element[0].strip()

    if len(list_of_element) == 0:
        return "Not Specified"
    
    return category_name


df_elite = df_categories.withColumn("Category_name", get_category_name(F.col("categories"))).drop(F.col("categories"))
df_elite_2 =df_elite.select("Category_name").distinct()
df_elite_3 = df_elite_2.withColumn("Category_pk_sk", F.monotonically_increasing_id() + 1)






In [7]:
df_elite_3.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_category")